# Benchmark גניבה / ונדליזם על Colab (UCF-Crime Anomaly-Videos-Part-4)

מאנדקס את 200 הסרטונים על GPU ומודד precision / recall / F1 לתרחישים `conceal`, `vandal`, `vehicle`.

**לפני שמתחילים (פעם אחת):** להעלות את `Anomaly-Videos-Part-4.zip` (~6GB) ל-Google Drive, לתיקייה `MyDrive/ucf_bench/`.

**הרצה:** Runtime → Change runtime type → **T4 GPU**, ואז להריץ את התאים לפי הסדר.

**התקדמות נשמרת ב-Drive** (`MyDrive/ucf_bench/benchmark.db`) אחרי כל סרטון. אם Colab מתנתק, פשוט להריץ שוב את כל התאים: סרטונים שכבר אונדקסו מדולגים.

In [ ]:
# 1. GPU + files from a shared Drive folder link (no Google sign-in needed)
!nvidia-smi --query-gpu=name,memory.total --format=csv
import os, glob, shutil
FOLDER_URL = 'https://drive.google.com/drive/folders/12Ga42OWJD2DkHy5tZjoClNk4ETd0JImy?usp=drive_link'
DRIVE = '/content/ucf_bench'
os.makedirs(DRIVE, exist_ok=True)
ZIP = f'{DRIVE}/Anomaly-Videos-Part-4.zip'
if not os.path.exists(ZIP):
    !pip install -q -U gdown
    !gdown --folder "{FOLDER_URL}" -O /content/gdown_tmp
    for p in glob.glob('/content/gdown_tmp/**/*', recursive=True):
        if os.path.isfile(p):
            shutil.move(p, os.path.join(DRIVE, os.path.basename(p)))
print(sorted(os.listdir(DRIVE)))
print('zip found:', os.path.exists(ZIP))

name, memory.total [MiB]
Tesla T4, 15360 MiB
Retrieving folder contents
Processing file 1reUeI8mitlG4Q2JCg-XbTHZAde8MfCMh Anomaly-Videos-Part-4.zip
Processing file 1xDt3Z95JAQqI1l4PnV3xy6vfw3whCETL benchmark.db
Processing file 1A1LTj0XBLz04xW0_IJXdwT-HcBKnilRw indexed.txt
Processing file 1v6cyBsmmp2v7KtM5OtvLoMS7aViTj1XC part4_eval.txt
Processing file 12ugCkDLaN98d7fghCehafo7HXuSdYqmQ part4_report.json
Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From (original): https://drive.google.com/uc?id=1reUeI8mitlG4Q2JCg-XbTHZAde8MfCMh
From (redirected): https://drive.google.com/uc?id=1reUeI8mitlG4Q2JCg-XbTHZAde8MfCMh&confirm=t&uuid=5467265e-3367-467d-9ea8-af410cabcb18
To: /content/gdown_tmp/Anomaly-Videos-Part-4.zip
 80% 5.28G/6.56G [01:08<00:27, 46.3MB/s]

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Code + dependencies (Colab already has torch with CUDA)
BRANCH = 'ucf-part4-benchmark'   # the branch with the part-4 scenarios (or 'main' once merged)
!rm -rf /content/final_project
!git clone -q -b {BRANCH} https://github.com/Talyaavraham/final_project.git /content/final_project
%cd /content/final_project/backend
!pip install -q ultralytics lap fastapi "uvicorn[standard]" python-multipart httpx imageio-ffmpeg
!ls *.pt eval/uca_ground_truth.py

In [ ]:
# 3. Videos: unzip from Drive to the local disk, flat (eval/data/ucf/<name>.mp4)
import zipfile, shutil
UCF = 'eval/data/ucf'
os.makedirs(UCF, exist_ok=True)
with zipfile.ZipFile(ZIP) as z:
    members = [m for m in z.namelist() if m.endswith('.mp4')]
    for m in members:
        dst = os.path.join(UCF, os.path.basename(m))
        if not os.path.exists(dst):
            with z.open(m) as src, open(dst, 'wb') as f:
                shutil.copyfileobj(src, f, 8 << 20)
print(len(os.listdir(UCF)), 'videos')

In [ ]:
# 4. UCA annotations -> uca_all.json, video selection, ground truth
import json, glob, urllib.request
BASE = 'https://raw.githubusercontent.com/Xuange923/Surveillance-Video-Understanding/main/UCF%20Annotation/json/'
uca = {}
for split in ('Train', 'Val', 'Test'):
    uca.update(json.load(urllib.request.urlopen(BASE + f'UCFCrime_{split}.json')))
json.dump(uca, open('eval/data/uca_all.json', 'w', encoding='utf-8'))
sel = {os.path.basename(p)[:-4]: {'bytes': os.path.getsize(p)} for p in sorted(glob.glob(f'{UCF}/*.mp4'))}
json.dump(sel, open('eval/data/ucf_selection_part4.json', 'w'), indent=1)
!python eval/uca_ground_truth.py --sel eval/data/ucf_selection_part4.json --scenarios conceal,vandal,vehicle --out eval/ground_truth_uca_part4.json | grep -v '^    '

In [ ]:
# 5. Start the backend (same settings as eval/run_benchmark.ps1), resuming the index from Drive
import subprocess, time, httpx, sqlite3
DB = '/content/benchmark.db'                 # local: SQLite on the Drive mount is slow and unsafe
if os.path.exists(f'{DRIVE}/benchmark.db') and not os.path.exists(DB):
    shutil.copy(f'{DRIVE}/benchmark.db', DB)
    print('resumed index from Drive')
env = dict(os.environ, YOLO_MODEL='yolov8n.pt,night.pt', EXTRA_MODELS='security.pt', IMGSZ='800',
           CLIP_RERANK='1', DB_PATH=DB, VIDEO_DIR='/content/benchmark_videos')
server = subprocess.Popen(['python', '-m', 'uvicorn', 'main:app', '--port', '8011'], env=env,
                          stdout=open('/content/server.log', 'w'), stderr=subprocess.STDOUT)
API = 'http://127.0.0.1:8011'
for _ in range(90):
    try:
        print(httpx.get(f'{API}/api', timeout=5).json()); break
    except Exception:
        time.sleep(2)
else:
    print(open('/content/server.log').read()[-3000:])

def save_to_drive():
    src, dst = sqlite3.connect(DB), sqlite3.connect('/content/benchmark_copy.db')
    src.backup(dst); dst.close(); src.close()          # consistent copy while the server writes
    shutil.copy('/content/benchmark_copy.db', f'{DRIVE}/benchmark.db')

In [ ]:
# 6. Index in batches. Run this cell again for the next batch.
import sys, random
sys.path.insert(0, 'eval')
from index_videos import index_one
BATCH = 175
files = sorted(glob.glob(f'{UCF}/*.mp4'))
random.Random(0).shuffle(files)             # mixed order: every batch has Shoplifting, Stealing and Vandalism
done_file = f'{DRIVE}/indexed.txt'
done = set(open(done_file).read().split()) if os.path.exists(done_file) else set()
todo = [f for f in files if os.path.basename(f) not in done][:BATCH]
print(f'{len(done)} indexed, this batch: {len(todo)}')
t_all = time.time()
for i, p in enumerate(todo, 1):
    try:
        name, model, secs = index_one(API, p)
        print(f'[{i}/{len(todo)}] {name}: {model} ({secs:.0f}s)', flush=True)
        if not str(model).startswith(('ERROR', 'TIMEOUT')):
            done.add(os.path.basename(p))
            open(done_file, 'w').write(chr(10).join(sorted(done)))
            save_to_drive()
    except Exception as e:
        print(f'[{i}/{len(todo)}] {os.path.basename(p)}: FAILED {e}', flush=True)
print(f'batch done in {(time.time()-t_all)/60:.1f} min; {len(done)}/{len(files)} indexed')

In [ ]:
# 7. Score everything indexed so far (no server needed, seconds). The report is saved to Drive.
!python eval/offline_eval.py {DB} eval/ground_truth_uca_part4.json --report {DRIVE}/part4_report.json | tee {DRIVE}/part4_eval.txt

**כיול ספים** (אחרי שיש מספיק סרטונים), למשל:

```
!python eval/offline_eval.py {DB} eval/ground_truth_uca_part4.json --set MIN_SCENE=0.3 --set SCENE_Z=2
!python eval/sweep_scene.py {DB} eval/ground_truth_uca_part4.json
```

את `benchmark.db` מ-Drive אפשר גם להוריד למחשב ולהריץ עליו `offline_eval.py` מקומית.